## Networking in Kubernetes
Each pod in Kubernetes cluster gets a unique IP address as shown below:  
![IP Addresses](./images/cluster_ips.png)

To be able to assign unique IP addresses, each node is assigned a non-overlapping CIDR block. Example: `10.244.1.0/24` for node 1, `10.244.2.0/24` for node 2.

All containers in a pod share the same namespace, thus one container can talk to another over *localhost* (1). Pods on the same node reach each other through a virtual Ethernet pair (`veth`) plugged into a local software bridge (`docker0`) (2).

How does a packet travel from one pod in node A to another pod in node B? It uses CNI machinery. There are two layers of addressing here - `192.168.1.10 -> 192.168.1.11` and `10.244.0.3 -> 10.244.0.4`.

## Service
Pods come and go. With it the IP addresses of pods also change. Thus using IP address for communication between pods is not ideal. Here is where Service come into picture. Service provides stable IP/DNS name in front of a set of pods whose individual IPs keep changing. The convention is one service per deployment. A pod can expose multiple ports on which it listens to, a service however maps to only one of it.

When we apply the service definition:
- `api-server`:
  - picks virtual IP address `10.96.102.20` at which this service would be available at
  - saves to `etcd`
  - **CoreDNS** creates an entry like `http://order-service -> 10.96.102.20`
- `kube-controller-manager` has a component *EndpointSlice controller* that maintains list of all pods and their IP addresses.
- `kube-proxy` polls `api-server` and
   - obtains service and its pods list.
   - asks kernel to update its route table such that when a packet it sent to `10.96.102.20` replace it with the IPs of the underlying pods at random. This is how requests are load balanced and in this case it is not *round-robin*.

![ClusterIP Mode](./images/cluster_ip.png)

When pod A makes a call to `http://order-service`:
1. CoreDNS translates that name to `10.96.0.20`
2. Pod A sends the network packet toward `10.96.0.20`
3. Before the packet even leaves the server, the Linux kernel intercepts it and swaps the destination IP to a real pod IP (`10.244.2.8`).

A service can be one of the following types:
1. `ClusterIP`: only reachable from inside the cluster. This is what we use for internal service-to-service traffic, e.g. search service calling payments service. Not reachable from an external source.
2. `NodePort`: everything `ClusterIP` does, plus it opens a specific port (30000–32767 range) on every node's actual IP. So `<any-node-ip>:<nodePort>` reaches your Service from outside the cluster.  
   ![NodePort](./images/node_port.png)
   
3. `LoadBalancer`: everything `NodePort` does, plus it asks the underlying cloud provider (AWS, GCP, etc.) to provision a real external load balancer with a clean public IP pointing at it.

## Service Configurations
A simple `ClusterIP` service configuration looks like:
```yaml
apiVersion: v1
kind: Service
metadata:
  name: api-service
  namespace: default
spec:
  type: ClusterIP
  selector:
    app: backend-api # Must match pod labels
  ports:
    - name: http
      protocol: TCP
      port: 80               # Port exposed by the Service internally
      targetPort: 8080       # Port the container process listens on
```

`port` is the port that other containers in the cluster call. Example: `http://api-service:80`. `targetPort` is the actual port the application is listening on inside the container.

`NodePort` configuration:
```yaml
apiVersion: v1
kind: Service
metadata:
  name: nodeport-service
spec:
  type: NodePort
  selector:
    app: backend-api
  ports:
    - port: 80               
      targetPort: 8080       
      nodePort: 31080   # Optional fixed port (30000-32767). If omitted, k8s auto-assigns.
```

Service and deployment together can be used to do canary deployments:
```yaml
apiVersion: v1
kind: Service
metadata:
  name: api-service
spec:
  type: ClusterIP
  selector:
    app: api          # Matches BOTH stable and canary Pods
  ports:
    - port: 80
      targetPort: 8080

---
# Stable
apiVersion: apps/v1
kind: Deployment
metadata:
  name: api-stable
spec:
  replicas: 9
  selector:
    matchLabels:
      app: api
      track: stable
  template:
    metadata:
      labels:
        app: api     
        track: stable
    spec:
      containers:
        - name: api
          image: myorg/api:v1.0.0
          ports:
            - containerPort: 8080

---
# Canary
apiVersion: apps/v1
kind: Deployment
metadata:
  name: api-canary
spec:
  replicas: 1
  selector:
    matchLabels:
      app: api
      track: canary
  template:
    metadata:
      labels:
        app: api     
        track: canary
    spec:
      containers:
        - name: api
          image: myorg/api:v2.0.0
          ports:
            - containerPort: 8080
```